<a href="https://colab.research.google.com/github/IOAI-official/IOAI-2026/blob/main/Home%20Task/Home-Task-3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🗄️ The Analytical Language of John Wilkins

> *"These ambiguities, redundancies, and deficiencies recall those attributed by Dr. Franz Kuhn to a certain Chinese encyclopedia called the *Celestial Emporium of Benevolent Knowledge*. On those remote pages it is written that animals are divided into (a) those that belong to the Emperor, (b) embalmed ones, (c) those that are trained, (d) suckling pigs, (e) mermaids, (f) fabulous ones, (g) stray dogs, (h) those that are included in this classification..."*
>
> — Jorge Luis Borges, *The Analytical Language of John Wilkins*

## The story

In the seventeenth century a churchman named John Wilkins set out to build a perfect language — one in which the very *spelling* of a word would declare the nature of the thing it named. Each animal would be filed under a rigorous tree of yes-and-no distinctions: beast or fish, winged or finned, tame or wild, until the creature stood alone at the end of a single branch, named by the path that led to it.

The scheme failed, as all such schemes fail. But somewhere a clerk kept building it anyway. He bound every animal of the world into a great **Cabinet of Distinctions** — and then, before the index could be written, he died. The drawers remain. Each holds one creature behind a small brass grille, and the creature will not say its name. It will only answer **yes** or **no** to questions about its own nature.

The Cabinet has come to you with its labels lost. Two lists survived in the clerk's hand:

- `animals_pool.txt` — every creature filed in the Cabinet (~1,400 entries).
- `questions_pool.txt` — every distinction the clerk thought to draw (~500 yes/no questions).

Open a drawer. Ask your distinctions. Find the path that names the beast.

## Portable execution layer

This tracked copy preserves the latest local work and its saved evidence. Downloads are verified against `manifest.json` and stored only in this task's ignored `.data/` and `.cache/` directories. Generated files go to `outputs/`. Set `PORTABLE_IOAI_SMOKE=1` to execute only the small CPU portability contract; the original workload cells are tagged `full-run`.


In [ ]:
from pathlib import Path
import json
import os
import sys

def _find_portable_root():
    start = Path.cwd().resolve()
    for parent in (start, *start.parents):
        for candidate in (parent, parent / 'olympiads' / 'portable_ioai'):
            if (candidate / 'bootstrap.py').is_file() and (candidate / 'manifest.json').is_file():
                return candidate
    raise FileNotFoundError(
        'Could not locate olympiads/portable_ioai. Start Jupyter from the '
        'repository root or this notebook directory, after running setup.ps1.'
    )

PORTABLE_ROOT = _find_portable_root()
if str(PORTABLE_ROOT) not in sys.path:
    sys.path.insert(0, str(PORTABLE_ROOT))
from bootstrap import NotebookContext, load_hf_datasets, smoke_task

PORTABLE = NotebookContext('home_task_3').prepare_paths()
DATA_DIR = PORTABLE.data_dir
OUTPUT_DIR = PORTABLE.output_dir
SMOKE_MODE = os.environ.get('PORTABLE_IOAI_SMOKE', '').lower() in {'1', 'true', 'yes'}
os.environ[PORTABLE.spec['data_environment']] = str(DATA_DIR)
if 'home_task_3' == 'home_task_3':
    model_mode = 'smoke' if SMOKE_MODE else 'full'
    model_spec = PORTABLE.spec['models'][model_mode]
    os.environ['PORTABLE_IOAI_HT3_MODEL'] = model_spec['id']
    os.environ['PORTABLE_IOAI_HT3_MODEL_REVISION'] = model_spec['revision']
print(json.dumps({**PORTABLE.describe(), 'smoke_mode': SMOKE_MODE}, indent=2))


In [ ]:
if SMOKE_MODE:
    SMOKE_RESULT = smoke_task('home_task_3', ensure=True)
    print(json.dumps(SMOKE_RESULT, indent=2, default=str))
else:
    print('Portable context ready; continuing with the preserved full-workload cells.')


## Your task

Each hidden creature sits inside a sealed oracle called an **Interactor** — a brass grille over a drawer, holding one animal. You cannot see it. You may put to it one of two kinds of question:

| Call | Returns | What you are asking |
|---|---|---|
| `interactor.ask(question)` | `"yes"` or `"no"` | A yes/no question about the hidden animal. The question must be a line from `questions_pool.txt`. |
| `interactor.guess(animal)` | `"correct"` or `"wrong"` | "Is this the hidden animal?" `"correct"` ends the row. The animal must be a word from `animals_pool.txt`. |

Each question in the pool refers to the creature generically — *"is it a mammal?"*, *"does it live in water?"*, *"can it fly?"* — and the oracle answers about whichever animal is hidden in that drawer.

If you submit a question or animal not in the relevant pool, the oracle refuses without spending its strength: a `ValueError` is raised and your budget is unchanged. Typos cost nothing.

Each drawer will entertain at most **fifteen questions** before the grille falls shut.

### Scoring

For each creature:

```
score = max(0, (1 if you ever guessed correctly else 0) - 0.02 × queries_used)
```

- Correct guess on question 1 → 0.98
- Correct guess on question 5 → 0.90
- Correct guess on question 15 → 0.70
- Never correct → 0

Your score is the mean across all creatures in a test set. Tune on `dev`, then run the final cell to get your **`test1`** score — that summary table is what you submit a screenshot of. The organizers keep a second, **hidden** test set for official grading, so a solution that genuinely deduces (rather than overfits `dev`/`test1`) is what scores well.

## The oracle

In plain language, the oracle inside each Interactor is a local language model — by default, `Qwen/Qwen2.5-3B-Instruct`. When you call `ask(question)` it prompts the model with:

```
You are answering a question about one specific animal.
The animal is: <hidden animal>.
Answer with a single word, yes or no.
Question: <question>
```

at temperature 0, parses the first word of the reply, and returns `"yes"` or `"no"` to your code.

The model is deterministic (the same `(animal, question)` pair always gives the same answer) and runs entirely inside the Interactor. You are free to run the same model in your own code to **predict** what it will say without spending the oracle's strength — that is a large part of what makes a clever solution. Note the oracle answers from the model's *beliefs* about the animal, which are usually right but not infallible; a good solution is robust to the occasional surprising answer.

## Step 1: Setup

The dataset and helper code (`interactor.py`, `evaluate.py`, the two pools, the dev/test CSVs) live in the shared **`IOAI-2026/AnimalDeduction/dataset`** Drive folder. The cell below just downloads them into Colab — no sign-in, no shortcuts, just run it. Use a **GPU** runtime: *Runtime → Change runtime type → T4* (free tier is enough).

In [ ]:
import os
import sys
from pathlib import Path

LOCAL_DIR = PORTABLE.ensure_data()
SUPPORT_DIR = PORTABLE.task_dir / "support"
if str(SUPPORT_DIR) not in sys.path:
    sys.path.insert(0, str(SUPPORT_DIR))
os.environ["PORTABLE_IOAI_HOME_TASK_3_DATA"] = str(LOCAL_DIR)
print("Data directory:", LOCAL_DIR)
print("Tracked helpers:", sorted(path.name for path in SUPPORT_DIR.iterdir()))
print("Data files:", sorted(path.name for path in LOCAL_DIR.iterdir()))


In [ ]:
LOCAL_DIR


## Step 2: Load data and try the oracle

The `Interactor` owns the hidden gold animal and runs a local LLM (Qwen 2.5 3B Instruct by default) to answer yes/no questions about it. The first `Interactor(...)` instantiation triggers the LLM download (~6 GB on first run, takes 30-60 s on T4). Every subsequent Interactor reuses the same LLM that's already loaded in memory.

In [ ]:
import random
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

from interactor import Interactor
from evaluate import evaluate, load_pools

DEVICE = "cpu" if SMOKE_MODE else ("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

animals_pool, questions_pool = load_pools(
    LOCAL_DIR / "animals_pool.txt",
    LOCAL_DIR / "questions_pool.txt",
)
print(f"animals_pool size:   {len(animals_pool):>6}  (e.g. {animals_pool[:5]})")
print(f"questions_pool size: {len(questions_pool):>6}  (e.g. {questions_pool[:3]})")

probe = Interactor(
    gold_animal="octopus",
    animals_pool=animals_pool,
    questions_pool=questions_pool,
)
probe_questions = questions_pool[:1] if SMOKE_MODE else [
    "is it a mammal?",
    "does it live in water?",
]
for question in probe_questions:
    print(f"ask({question!r}) ->", probe.ask(question))
print("Queries used:", probe.queries_used, "/", probe.budget)

if torch.cuda.is_available() and next(Interactor._model.parameters()).dtype != torch.float16:
    Interactor._model = Interactor._model.half()
    print("oracle model -> float16")


## Step 3: Solution interface

Your solution is a class with two methods:

- `__init__(self, animals_pool, questions_pool)` — runs once. Load models, precompute tables, etc.
- `solve(self, interactor)` — runs once per test row. Use the oracle to identify the hidden animal.

Inside `solve`, you have:

```
interactor.ask(question)      -> 'yes' or 'no'        (question must be in questions_pool)
interactor.guess(animal)      -> 'correct' or 'wrong' (animal must be in animals_pool)
interactor.is_done()          -> True after a correct guess or budget exhausted
interactor.remaining_budget() -> int
```

**Scoring per row**: `score = max(0, (1 if you ever guess correctly else 0) - 0.02 * total_queries)`.

Budget is **15 questions** per row. Information theory: `log₂(1400) ≈ 10.5 bits`, each yes/no answer is at most 1 bit — so ~11 well-chosen questions plus 1 final guess fit the budget, *if* each question splits the remaining candidates in half. Most don't: *"does it have a backbone?"* sounds decisive but the model calls a great many creatures vertebrates. A good question splits the *remaining* candidates roughly in half, given everything you have already learned — so the right next question depends on the answers so far.

### Baseline: random guessing (the floor)

Ignores `ask()` entirely. Just guesses random animals until the budget runs out. Expected score: ~0 (15 random guesses out of ~1,400 candidates ≈ 1% solve rate). Any reasonable solution needs to beat this by a lot.

In [ ]:
class RandomBaseline:
    def __init__(self, animals_pool, questions_pool, seed=0):
        self.animals_pool = animals_pool
        self.questions_pool = questions_pool
        self.rng = random.Random(seed)

    def solve(self, interactor):
        guessed = set()
        while not interactor.is_done():
            cand = self.rng.choice(self.animals_pool)
            while cand in guessed:
                cand = self.rng.choice(self.animals_pool)
            guessed.add(cand)
            interactor.guess(cand)

baseline_results = evaluate(RandomBaseline(animals_pool, questions_pool), 'dev.csv')

### Reference: a non-adaptive 20-questions sketch

This reference shows the *shape* of a real solution without giving away the points. In `__init__` it precomputes, with its own copy of the model, the oracle's yes/no answer to a small **fixed** list of broad questions for every animal — a bit-vector per animal. In `solve` it asks those same fixed questions, reads off the oracle's bit-vector, and guesses the animals whose precomputed vector is closest.

It works, but it's deliberately weak: the questions are the **same for every row** (not chosen adaptively to split the *remaining* candidates), and it uses only a handful. Beating it is mostly about (1) precomputing the full `animal × question` table and (2) choosing each next question *greedily* to most evenly split the animals still consistent with the answers so far. That's your job in Step 4.

> Precomputing even this small table calls the model a few thousand times (~5-15 min on T4). Skip this cell if you just want to get to your own solution — it is only a reference.

### 💡 Speed tip

Building the animal×question table by calling `interactor.ask()` **one at a time** is slow.
You can make your precompute **much** faster — without changing any answers — by **batching**
your model calls (many prompts through the model per forward pass). That optimization is up to you.

In [ ]:
import re
import torch
from interactor import Interactor, JUDGE_PROMPT

def parse_yes_no(raw):
    clean = re.sub(r"[^a-z]", "", raw.lower())
    if clean.startswith("yes"):
        return 1
    if clean.startswith("no"):
        return 0
    y, n = clean.find("yes"), clean.find("no")
    if y == -1:
        return 0
    if n == -1:
        return 1
    return int(y < n)

def batch_judge(animal_question_pairs, batch_size=32):
    Interactor._ensure_llm()
    tok = Interactor._tokenizer
    model = Interactor._model

    tok.padding_side = "left"
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token

    answers = []
    print(f"animal-queston pairs count: {len(animal_question_pairs)}")
    for i in tqdm(range(0, len(animal_question_pairs), batch_size)):
        batch = animal_question_pairs[i:i + batch_size]
        # print(batch)
        chats = []
        for animal, question in batch:
            prompt = JUDGE_PROMPT.format(animal=animal, question=question)
            messages = [{"role": "user", "content": prompt}]
            chat = tok.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
            chats.append(chat)

        inputs = tok(chats, return_tensors="pt", padding=True).to(model.device)

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=5,
                do_sample=False,
                pad_token_id=tok.eos_token_id,
            )

        prompt_len = inputs.input_ids.shape[1]
        for row in out:
            raw = tok.decode(row[prompt_len:], skip_special_tokens=True)
            answers.append(parse_yes_no(raw))

    return answers

In [ ]:
# Reference solution (optional, slow to init). Demonstrates precompute + match,
# but uses FIXED, non-adaptive questions -> leaves most of the score on the table.
FIXED_QUESTIONS = [
    'is it a mammal?',
    'is it a bird?',
    'is it a fish?',
    'is it an insect?',
    'does it live in water?',
    'can it fly?',
    'is it a carnivore?',
    'is it bigger than a human?',
    'does it have a backbone?',
    'is it commonly kept as a pet?',
    'does it have legs?',
    'does it lay eggs?',
]

class FixedQuestionsReference:
    def __init__(self, animals_pool, questions_pool, max_animals=None):
        self.animals_pool = animals_pool
        self.questions_pool = set(questions_pool)
        self.fixed = [q for q in FIXED_QUESTIONS if q in self.questions_pool]
        
        # new_fixed = []
        # for i in range(0, len(self.fixed), 2):
        #     if i + 1 < len(self.fixed):
        #         new_fixed.append(f"{self.fixed[i]};{self.fixed[i+1]}")
        #     else:
        #         new_fixed.append(self.fixed[i])  # Handle odd number of elements
        # self.fixed = new_fixed
        
        from interactor import Interactor
        cand = animals_pool if max_animals is None else animals_pool[:max_animals]
        self.candidates = cand
        print(f'  [reference] precomputing {len(cand)} x {len(self.fixed)} answer table...')
        self.table = {}
        for i, a in enumerate(tqdm(cand)):
            sim = Interactor(gold_animal=a, animals_pool=self.animals_pool,
                             questions_pool=self.questions_pool, budget=10**9)
            self.table[a] = tuple(1 if sim.ask(q) == 'yes' else 0 for q in self.fixed)
            if (i + 1) % 200 == 0:
                print(f'    {i+1}/{len(cand)}')
        print('  [reference] table ready.')

    def solve(self, interactor):
        obs = []
        for q in self.fixed:
            if interactor.remaining_budget() <= 1:
                break
            obs.append(1 if interactor.ask(q) == 'yes' else 0)
        obs = tuple(obs)
        def agree(a):
            vec = self.table[a]
            return sum(1 for x, y in zip(vec, obs) if x == y)
        ranked = sorted(self.candidates, key=agree, reverse=True)
        for a in ranked:
            if interactor.is_done():
                break
            interactor.guess(a)
            

# Example (commented out by default — uncomment to run; slow to init):
# ref = FixedQuestionsReference(animals_pool, questions_pool)
# ref_results = evaluate(ref, 'dev.csv')

## Step 4: Your solution

Replace the body of `MySolution.solve` (and `__init__` if you precompute anything) with your strategy. Iterate on `dev.csv` until you're happy with the score, then jump to Step 5 to evaluate on test1 + test2.

**The intended approach:**
1. In `__init__`, precompute once — with your own copy of the model — the oracle's yes/no answer for every `(animal, question)` pair you care about. This costs no oracle budget.
2. In `solve`, keep a set of candidate animals still consistent with the answers so far. At each step pick the **question whose answer most evenly splits that set** (maximize information gain), ask it, and shrink the set. Guess when one candidate dominates or the budget is nearly gone.
3. Be robust: the oracle occasionally answers in a way your table didn't predict. Don't let one surprising bit eliminate the true animal forever.

In [ ]:
def build_decision_tree(res, depth=0):
    """
    Recursively builds a decision tree structure.

    Args:
        res (np.array): The data matrix [animals, questions].

    Returns:
        dict: A dictionary representing the node. 
              If it's a leaf, it might contain the classification (e.g., a count or label).
              If it's an internal node, it contains 'question_index' and 'children'.
    """
    # Base case: If all samples belong to the same class (or if only one sample remains)
    # In this binary classification context (Yes/No), if all the remaining rows are 1s or all are 0s, we stop.
    # Or, if only one sample remains.
    if res.shape[0] == 0:
        return None  # No samples left
    
    # Check for purity (All outcomes are the same)
    class_sums = res.sum(axis=0)
    all_ones = np.all(res == 1)
    all_zeros = np.all(res == 0)
    
    # If all remaining rows are the same (e.g., all 'Yes' or all 'No' for the measured outcome)
    # We stop and declare a leaf node, though in this setup, the purity check is implied by the stopping condition below.

    # Find the best question (feature) to split on (Minimize Gini/Entropy or use the existing Heuristic)
    q = np.abs((res.sum(axis=0) / res.shape[0]) - 0.5).argmin()
    
    # Check if the split is meaningful (i.e., if the question results in at least one 'Yes' and one 'No')
    # If the split on this question does not change the composition significantly, we might stop early.
    if np.all(res[:, q] == 0) or np.all(res[:, q] == 1):
        # All current samples are the same on this question/feature, so we stop here.
        return {"leaf": True, "last_question": q}

    # Find the split
    mask_yes = res[:, q] == 1
    mask_no = ~mask_yes  # The "No" branch

    # Recursive calls
    left_child = build_decision_tree(res[mask_yes], depth + 1)
    right_child = build_decision_tree(res[mask_no], depth + 1)

    # Return the internal node structure
    return {
        "node_type": "internal",
        "question_index": q,
        "children": {
            "yes": left_child,   # Branch where res[:, q] == 1
            "no": right_child    # Branch where res[:, q] == 0
        }
    }

# Example Usage:
# Assuming your input dataset is 'res'
# tree_structure = build_decision_tree(res)

# print(tree_structure)

In [ ]:
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd

HOME3_MAX_QUESTIONS = 32      # Fast first pass: 1472 x 32 = 47,104 LLM calls, not 823,648.
HOME3_BATCH_SIZE = 32         # Safer on limited VRAM than 128.
HOME3_SMOKE_ROWS = 10         # Prove the whole path before full dev/test.

class MySolution:
    """Bounded greedy 20-questions style solver.

    Contest rule for today: keep all animals, reduce questions. Reducing animals
    makes dev/test impossible when the hidden animal is outside the subset.
    Questions are selected directly from questions_pool.txt, not hard-coded.
    """

    def __init__(self, animals_pool, questions_pool, max_questions=HOME3_MAX_QUESTIONS,
                 min_guess_budget=4, cache_path=None):
        self.animals_pool = list(animals_pool)
        self.questions_pool = list(questions_pool)
        self.min_guess_budget = min_guess_budget
        self.questions = self._select_questions(max_questions)
        if not self.questions:
            raise ValueError('questions_pool.txt did not provide any usable questions')

        if cache_path is None:
            cache_path = Path(f'home3_answer_table_{len(self.animals_pool)}x{len(self.questions)}.npz')
        self.cache_path = Path(cache_path)

        table = self._load_cached_table()
        if table is None:
            pair_count = len(self.animals_pool) * len(self.questions)
            print(f'  [greedy] precomputing {len(self.animals_pool)} animals x {len(self.questions)} questions = {pair_count} LLM calls')
            pairs = [(animal, question) for animal in self.animals_pool for question in self.questions]
            answers = batch_judge(pairs, batch_size=HOME3_BATCH_SIZE)
            table = np.array(answers, dtype=np.int8).reshape(len(self.animals_pool), len(self.questions))
            np.savez_compressed(
                self.cache_path,
                animals=np.array(self.animals_pool, dtype=object),
                questions=np.array(self.questions, dtype=object),
                table=table,
            )
            print(f'  [greedy] cached answer table -> {self.cache_path}')
        self.table = table

        name_counts = Counter(word for a in self.animals_pool for word in a.split())
        self.guess_order = sorted(
            range(len(self.animals_pool)),
            key=lambda i: (len(self.animals_pool[i].split()), -sum(name_counts[w] for w in self.animals_pool[i].split()), self.animals_pool[i])
        )

    def _select_questions(self, max_questions):
        """Pick a bounded, deterministic subset directly from questions_pool.txt."""
        seen = set()
        selected = []
        for question in self.questions_pool:
            q = str(question).strip().lower()
            if not q or q in seen:
                continue
            seen.add(q)
            selected.append(q)
            if len(selected) >= max_questions:
                break
        print(f'  [greedy] selected first {len(selected)} questions from questions_pool.txt')
        return selected

    def _load_cached_table(self):
        if not self.cache_path.exists():
            return None
        try:
            cached = np.load(self.cache_path, allow_pickle=True)
            same_animals = list(cached['animals']) == self.animals_pool
            same_questions = list(cached['questions']) == self.questions
            if same_animals and same_questions:
                print(f'  [greedy] loaded cached answer table <- {self.cache_path}')
                return cached['table'].astype(np.int8)
        except Exception as exc:
            print(f'  [greedy] ignoring stale cache {self.cache_path}: {type(exc).__name__}: {exc}')
        return None

    def _best_question(self, candidate_idx, used_questions):
        """Return the unused question that most evenly splits current candidates."""
        best_q = None
        best_balance = 10**9
        best_yes_count = None

        for q_idx in range(len(self.questions)):
            if q_idx in used_questions:
                continue
            yes_count = int(self.table[candidate_idx, q_idx].sum())
            no_count = len(candidate_idx) - yes_count
            if yes_count == 0 or no_count == 0:
                continue

            balance = abs(yes_count - no_count)
            if balance < best_balance:
                best_q = q_idx
                best_balance = balance
                best_yes_count = yes_count

        return best_q, best_yes_count

    def _rank_candidates_for_guessing(self, candidate_idx):
        remaining = set(int(i) for i in candidate_idx)
        ranked = [i for i in self.guess_order if i in remaining]
        return ranked if ranked else list(candidate_idx)

    def solve(self, interactor):
        candidate_idx = np.arange(len(self.animals_pool))
        used_questions = set()

        while (not interactor.is_done()
               and interactor.remaining_budget() > self.min_guess_budget
               and len(candidate_idx) > self.min_guess_budget):
            q_idx, yes_count = self._best_question(candidate_idx, used_questions)
            if q_idx is None:
                break

            used_questions.add(q_idx)
            answer = interactor.ask(self.questions[q_idx])
            wanted = 1 if answer == 'yes' else 0
            candidate_idx = candidate_idx[self.table[candidate_idx, q_idx] == wanted]

            if len(candidate_idx) == 0:
                candidate_idx = np.arange(len(self.animals_pool))
                break

        for i in self._rank_candidates_for_guessing(candidate_idx):
            if interactor.is_done():
                break
            interactor.guess(self.animals_pool[i])


# Fast path: build/load the bounded table, then smoke-test 10 dev rows first.
solution = MySolution(animals_pool, questions_pool)
_smoke_path = Path('dev_smoke_10.csv')
pd.read_csv('dev.csv').head(HOME3_SMOKE_ROWS).to_csv(_smoke_path, index=False)
smoke_results = evaluate(solution, _smoke_path)
print('SMOKE PASS: run the final dev/test cell next if this completed in acceptable time.')
smoke_results['per_row']


In [ ]:
def query_decision_tree(tree):
    """
    Traverses the decision tree and asks the user the required questions
    to reach a final classification (leaf node).
    Args:
        tree (dict): The decision tree structure returned by build_decision_tree.
    Returns:
        int or str: The predicted class/label from the leaf node, or None if the tree is empty.
    """
    if tree is None:
        return None
    if tree.get("leaf"):
        # Leaf node reached, return the result (here, we just return the last question index as a proxy for the result)
        return f"Prediction reached. The final node relates to question index: {tree.get('last_question')}"
    # Internal node, ask the question
    question_index = tree.get("question_index")
    
    # Construct a readable prompt based on the index. 
    # NOTE: In a real scenario, you would need a separate array linking question_index to the actual question text.
    prompt = f"Question #{question_index}: Does the sample satisfy this criterion? (Answer 1 for Yes, 0 for No)"

    # In a real system, we would call ask(prompt) here. 
    # For this demonstration, we assume the input answer is obtained and stored as 'user_answer'.
    # try: 
    #     # Simulate getting user input (must be 0 or 1 for this binary tree)
    #     user_input = int(input(prompt + " -> "))
    # except ValueError:
    #     print("Invalid input. Please enter 0 or 1.")
    #     return query_decision_tree(tree) # Retry or handle error
        
    
    # Traverse down the appropriate branch
    if user_input == 1:
        # Branch 'yes'
        next_node = tree["children"].get("yes")
    elif user_input == 0:
        # Branch 'no'
        next_node = tree["children"].get("no")
    else:
        # Should not happen if input validation works, but good practice to handle
        return "Error: Invalid branch choice."
    # Recurse down the tree
    return query_decision_tree(next_node)
# Example Usage (Requires a pre-built tree structure)
# Assuming 'decision_tree' is the output from build_decision_tree(res)
# print("\n--- Starting Decision Tree Query ---")
# result = query_decision_tree(decision_tree)
# print(f"\n--- Final Result ---")
# print(result)

In [ ]:
# res = np.array(solution.res).reshape(len(solution.animals_pool), len(solution.questions_pool))
# accs = abs((res.sum(axis=0) / res.shape[0]) - .5)
# questions_pool[accs.argmin()]
# res.shape

In [ ]:
# tree = build_decision_tree(res)
# query_decision_tree(tree)

Removed broken tree stub. The greedy `MySolution` above now handles asking and guessing directly.


In [ ]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("CUDA cache cleared.")
else:
    print("CUDA is not available.")
# del model

## Step 5: Final scoring

Once you're happy with your dev score, run this cell. It scores `dev` and `test1` (and `test2` automatically, if that file is present). The **FINAL** line — the *n*-weighted mean over the available test split(s) — is what you submit a screenshot of.

> The organizers also score your submitted `MySolution` on a separate **hidden** test set that is not included here. Aim for a strategy that deduces the animal from scratch each row, so it transfers to unseen creatures.

In [ ]:
import os

# Run this only after the smoke cell above completes.
dev_results = evaluate(solution, LOCAL_DIR / "dev.csv")
test1_results = evaluate(solution, LOCAL_DIR / "test1.csv")

splits = [("dev", dev_results), ("test1", test1_results)]
test2_path = LOCAL_DIR / "test2.csv"
if test2_path.exists():
    splits.append(("test2", evaluate(solution, test2_path)))

rows = [{
    "split": name, "n": result["n"], "mean_score": result["mean_score"],
    "solved_rate": result["solved_rate"], "mean_queries": result["mean_queries"],
} for name, result in splits]

tests = [result for name, result in splits if name.startswith("test")]
n_test = sum(result["n"] for result in tests)
rows.append({
    "split": "FINAL",
    "n": n_test,
    "mean_score": sum(result["mean_score"] * result["n"] for result in tests) / n_test,
    "solved_rate": sum(result["solved_rate"] * result["n"] for result in tests) / n_test,
    "mean_queries": sum(result["mean_queries"] * result["n"] for result in tests) / n_test,
})
pd.DataFrame(rows)
